# Aula 02 — Sintaxe Básica, Tipagem e Variáveis Econômicas
Semana 1 | 50 min | Referências: McKinney, Apêndice (*Python Language Essentials*) · VanderPlas, Cap. 1 (mágicos `%time`/`%timeit`)

## 🎯 Objetivos de aprendizagem
Ao final desta aula, você será capaz de:

- Ler e escrever código Python: indentação, comentários, `=` vs `==` e nomes idiomáticos;
- Distinguir os tipos essenciais (`int`, `float`, `str`, `bool`) e converter valores com segurança;
- Aplicar operadores aritméticos (`/`, `//`, `%`, `**`) e de atribuição (`+=`, `*=`) a cálculos econômicos;
- Escolher a estrutura nativa certa — lista, tupla ou dicionário — para cada problema de dados;
- Cronometrar código com `%time`/`%timeit` e explicar por que o NumPy vence o Python puro.

## 1. Tipos: cada grandeza econômica tem uma natureza
Em planilha, uma célula não diz se "1.800" é quantidade, preço ou código de
município — o Python sim. O **tipo** do valor governa o que se pode fazer com
ele: somar sacas faz sentido; somar "Goiânia" não. Os quatro tipos escalares
essenciais aparecem hoje em números reais do Brasil de 2026: câmbio ≈
R$ 5,1253/US$, IPCA acumulado de 12 meses ≈ 4,44%, soja ≈ US$ 12,9/bushel,
salário mínimo de R$ 1.518.

Quatro regras de semântica valem para todo o curso (McKinney, Apêndice):

1. **Indentação (4 espaços) define blocos** — não existem chaves `{}`;
2. **Tudo é objeto** — todo valor tem tipo e métodos (`x.tipo_de_coisa()`);
3. **Comentários** com `#` — comente a DECISÃO ("Selic 13,9% = set/2026"), não o óbvio;
4. **Case-sensitive** e `=` **atribui** (guarda), `==` **compara** (pergunta).

In [1]:
# Setup — imports e padrões visuais usados na aula
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

def brl(x):
    """Formata um número como moeda brasileira: R$ 1.234,56."""
    return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

np.random.seed(42)  # reprodutibilidade quando houver aleatoriedade
print("Setup concluído sem erros — ambiente operacional!")

Setup concluído sem erros — ambiente operacional!


In [2]:
# Cada grandeza econômica tem uma natureza — e o Python registra isso no tipo
sacas = 1_800               # int   — sacas de soja colhidas (quantidade inteira)
cambio = 5.1253             # float — R$/US$ (BCB-SGS 1, 04/09/2026)
municipio = "Goiânia"       # str   — texto, não número: somar código é absurdo
passa_na_tir = True         # bool  — "o projeto passa na TIR?" — decisão binária
for nome, valor in [("sacas", sacas), ("cambio", cambio),
                    ("municipio", municipio), ("passa_na_tir", passa_na_tir)]:
    print(f"{nome:<14} valor = {valor!r:<12} tipo = {type(valor).__name__}")

sacas          valor = 1800         tipo = int
cambio         valor = 5.1253       tipo = float
municipio      valor = 'Goiânia'    tipo = str
passa_na_tir   valor = True         tipo = bool


In [3]:
# Conversões explícitas evitam surpresas — e a maior surpresa é a aritmética do float
print(int(2.9))                    # 2 — trunca, não arredonda
print(float("1800"))               # texto → número (comum ao ler planilhas)
print(str(52))                     # número → texto (para imprimir/concatenar)
preco_texto = "5,12"               # vírgula decimal — padrão brasileiro
print(float(preco_texto.replace(",", ".")) + 0.1)   # 5.22 — converta ANTES de somar

# Armadilha clássica: 0.1 + 0.2 NÃO é exatamente 0.3 (representação binária)
print(0.1 + 0.2 == 0.3)                    # False — não é bug!
print(abs(0.1 + 0.2 - 0.3) < 1e-9)         # True — compare floats com tolerância
print(f"{0.1 + 0.2:.2f}")                  # exiba com 2 casas: 0.30
print(brl(145.7558))                       # moeda formatada — a soja da Seção 3

2
1800.0
52
5.22
False
True
0.30
R$ 145,76


## 2. Operadores aritméticos: a fórmula de planilha generalizada
Os operadores `+ - * /` você já conhece; os que pegam quem vem do Excel:

- `/` **sempre** devolve `float` — até `10/2` dá `5.0`, não `5`;
- `//` divisão inteira e `%` resto — quantas sacas inteiras e o que sobra;
- `**` potência — juros compostos: `(1 + i_mensal)**12` acumula o ano.

Na célula abaixo, cada operador atua sobre um número real do agro/finanças de
setembro/2026 (câmbio, soja, IPCA, Selic).

In [4]:
# Cada operador com um número real do agro/finanças (set/2026)
print("5.1253 * 12.9   =", 5.1253 * 12.9)       # câmbio × preço em US$ — conversão de moeda
print("12 / 5          =", 12 / 5)              # / SEMPRE devolve float — cuidado com 10/2 → 5.0
print("1800 // 60      =", 1800 // 60)          # 1.800 kg de soja → 30 sacas inteiras de 60 kg
print("1800 % 60       =", 1800 % 60)           # ... e nada sobra fora das sacas
print("(1.0044)**12    =", (1.0044)**12)        # IPCA 0,44%/mês capitalizado → ~5,4% a.a.
print("10000*1.139**5  =", 10_000 * 1.139**5)   # R$ 10 mil por 5 anos a Selic ~13,9% a.a.

5.1253 * 12.9   = 66.11637
12 / 5          = 2.4
1800 // 60      = 30
1800 % 60       = 0
(1.0044)**12    = 1.0540966873236182
10000*1.139**5  = 19169.84584049699


## 3. Aritmética aplicada: câmbio e agro
Duas contas que o economista faz toda semana, com dados reais de setembro/2026:

1. **Variação % do câmbio** entre dois pregões:
   $$ \Delta = \frac{c_t - c_{t-1}}{c_{t-1}} \times 100 $$
   (5,1273 em 02/09 → 5,0962 em 03/09/2026, série BCB-SGS 1);
2. **Preço da soja em reais**: US$/bushel → R$/saca de 60 kg —
   $$ P_{sc60kg} = P_{US\$/bu} \times \frac{60}{27{,}216} \times c $$
   com 1 bushel = 27,216 kg e o câmbio do dia.

O acumulado de inflação — produto dos $(1+i_k)$ — volta na Seção 6 com o IPCA
oficial lido do CSV.

In [5]:
# Variação percentual do câmbio em dois dias reais de setembro/2026 (BCB-SGS 1)
c_ontem = 5.1273             # 02/09/2026
c_hoje = 5.0962              # 03/09/2026 — dólar caiu
variacao = (c_hoje - c_ontem) / c_ontem * 100
print(f"Câmbio: {c_ontem} → {c_hoje} = {variacao:+.2f}% em um pregão")

Câmbio: 5.1273 → 5.0962 = -0.61% em um pregão


In [6]:
# Conversão agro: US$/bushel → R$/saca de 60 kg (soja)
preco_soja_usd = 12.9        # US$/bu — CBOT, set/2026
cambio = 5.1253              # R$/US$ — BCB-SGS 1 (04/09/2026)
preco_soja_sc = preco_soja_usd * (60 / 27.216) * cambio   # 1 bu = 27,216 kg
print(f"Soja: US$ {preco_soja_usd}/bu → {brl(preco_soja_sc)} por saca de 60 kg")

Soja: US$ 12.9/bu → R$ 145,76 por saca de 60 kg


## 4. Variáveis e atribuição composta: acumular custos, aplicar juros
Variável é um **nome apontando para um valor** — não uma caixa com tipo fixo.
Por isso `taxa` pode receber um float e depois uma string (o Python aceita;
cuidado apenas para o nome continuar fazendo sentido). As atribuições compostas
`+=` e `*=` são o dia a dia da modelagem: acumular custos e aplicar juros sem
reescrever a expressão inteira. Nomes idiomáticos do curso: `cambio`,
`preco_soja_usd`, `df`, `ax`.

In [7]:
# += acumula; *= aplica juros — as duas operações da rotina do economista
custo_ha = 0.0               # custo de produção de 1 ha de soja (R$)
custo_ha += 385.0            # sementes
custo_ha += 1250.0           # fertilizantes — maior rubro do custo
custo_ha += 290.0            # diesel e mecanização
print("Custo de produção por hectare:", brl(custo_ha))

# *= aplica juros: saldo corrigido em 1 ano de Selic (~13,9% a.a., set/2026)
saldo = 10_000.0
saldo *= 1.139
print("Saldo após 1 ano:", brl(saldo))

# Variável é um NOME, não uma caixa: o tipo pode ser reatribuído
taxa = 0.0444                # float — IPCA acumulado 12 meses
taxa = "4,44% a.a."          # agora str — legal, mas nomes devem significar algo
print(type(taxa), taxa)

Custo de produção por hectare: R$ 1.925,00
Saldo após 1 ano: R$ 11.390,00
<class 'str'> 4,44% a.a.


## 5. Listas, tuplas e dicionários: as três estruturas nativas
| Estrutura | Mutável? | Acesso | Uso típico no curso |
|---|---|---|---|
| lista (`list`) | sim | por posição — **índice começa em 0** | séries ordenadas — a "coluna de planilha" |
| tupla (`tuple`) | **não** | por posição | registro fixo — ex.: código IBGE + nome da UF |
| dicionário (`dict`) | sim | por **chave** | consulta por nome — o "PROCV mental" |

Fatiar com `ipca[:2]` pega os dois primeiros; `len()` conta elementos. Em
relatórios, o dict elimina o erro clássico de contar posição errada:
`precos["soja_sc"]` não depende da ordem das colunas.

In [8]:
# Lista = "coluna de planilha": mutável, indexada a partir de 0
meses = ["fev", "mar", "abr", "mai", "jun", "jul"]
ipca = [0.70, 0.88, 0.67, 0.58, 0.16, 0.07]   # % a.m., BCB-SGS 433 (fev–jul/2026)
print("mar/2026:", ipca[1])                   # índice começa em 0 — fev é ipca[0]!
ipca[1] = 0.90                                # mutável: dá para corrigir um valor
print("fev-abr corrigido:", ipca[:3], "| len:", len(ipca))

mar/2026: 0.88
fev-abr corrigido: [0.7, 0.9, 0.67] | len: 6


In [9]:
# Tupla = registro imutável: protege dados que não devem mudar
uf_go = (52, "Goiás")        # (código IBGE, nome) — UF de Goiás = 52
print(uf_go, "| nome:", uf_go[1])
try:
    uf_go[0] = 53            # tentativa de "consertar" o código IBGE por engano
except TypeError as erro:
    print("TypeError — tupla não muda:", erro)

(52, 'Goiás') | nome: Goiás
TypeError — tupla não muda: 'tuple' object does not support item assignment


In [10]:
# Dict = "PROCV mental": acesso por NOME, não por posição
precos = {"soja_sc": 12.9, "boi_at": 320.0, "cambio": 5.1253}   # US$/sc, R$/@, R$/US$
precos["milho_sc"] = 8.4     # adiciona/atualiza chave
print("câmbio:", precos["cambio"])
print("'boi_at' existe?", "boi_at" in precos)   # teste de chave antes de acessar
print("chaves:", list(precos.keys()))
print("café:", precos.get("cafe", "não monitorado"))   # .get: padrão se a chave falta

câmbio: 5.1253
'boi_at' existe? True
chaves: ['soja_sc', 'boi_at', 'cambio', 'milho_sc']
café: não monitorado


## 6. IPCA e câmbio de verdade (primeira degustação de pandas)
O que você aprendeu até aqui já opera sobre dados de verdade. O arquivo oficial
do BCB usa padrão brasileiro — separador `;` e decimal `,` — e o pandas lê isso
com `sep=";"` e `decimal=","`. O pandas em si é o tema das aulas 9-10; aqui é
só a degustação. O acumulado usa a fórmula do juro composto aplicada à série:

$$ i_{acum} = \left[\prod_{k=1}^{n}\left(1+\frac{i_k}{100}\right)\right] - 1 $$

— o mesmo cálculo do Excel `PRODUTO(1+A1:A12/100)-1`, em uma linha.

In [11]:
# Leitura do IPCA oficial (BCB-SGS 433): sep ';' e decimal ',' — padrão BR
from pathlib import Path
DATA = Path("../data/csv")   # notebook mora em semana_01/ — caminho relativo à pasta da semana

ipca_df = pd.read_csv(DATA / "ipca_mensal.csv", sep=";", decimal=",")
ipca_df["data"] = pd.to_datetime(ipca_df["data"], format="%d/%m/%Y", dayfirst=True)
ultimos_12 = ipca_df.tail(12)                # 12 meses mais recentes: ago/2025 → jul/2026
ultimos_12

,data,valor
547,2025-08-01,-0.11
548,2025-09-01,0.48
549,2025-10-01,0.09
550,2025-11-01,0.18
551,2025-12-01,0.33
552,2026-01-01,0.33
553,2026-02-01,0.70
554,2026-03-01,0.88
555,2026-04-01,0.67
556,2026-05-01,0.58


In [12]:
# Acumulado 12m: produto de (1 + i/100) — a fórmula da Seção 3 em uma linha
acumulado_12m = (1 + ultimos_12["valor"] / 100).prod() - 1
print(f"IPCA acumulado 12 meses (ago/2025 → jul/2026): {acumulado_12m * 100:.2f}%")

# A mesma fórmula em 6 meses (fev → jul/2026), sem pandas — deve dar ≈ 3,10%
acumulado_6m = 1.0
for i in [0.70, 0.88, 0.67, 0.58, 0.16, 0.07]:
    acumulado_6m *= (1 + i / 100)
print(f"IPCA acumulado 6 meses (fev → jul/2026): {(acumulado_6m - 1) * 100:.2f}%")

IPCA acumulado 12 meses (ago/2025 → jul/2026): 4.44%
IPCA acumulado 6 meses (fev → jul/2026): 3.10%


In [13]:
# Câmbio diário oficial (BCB-SGS 1): variação % entre os dois últimos pregões
usdbrl = pd.read_csv(DATA / "usdbrl_diario.csv", sep=";", decimal=",")
usdbrl["data"] = pd.to_datetime(usdbrl["data"], format="%d/%m/%Y", dayfirst=True)
print(usdbrl.tail(3).to_string(index=False))
delta_csv = (usdbrl["valor"].iloc[-1] / usdbrl["valor"].iloc[-2] - 1) * 100
print(f"Variação diária (últimos dois pregões): {delta_csv:+.2f}%")

      data  valor
2026-09-02 5.1273
2026-09-03 5.0962
2026-09-04 5.1253
Variação diária (últimos dois pregões): +0.57%


## 7. `%time` vs `%timeit` revisitados — 1 milhão de observações
Mesmo experimento da Aula 01, agora com 10⁶ observações: somar os inteiros de
0 a 999.999. `%time` roda a instrução **uma vez** (bom para operações longas);
`%timeit` repete e reporta a melhor média (bom para operações rápidas).
Hipótese: `np.sum` sobre um array NumPy vence `sum()` do Python puro por uma
ordem de grandeza ou mais — medir antes de otimizar é a regra do curso.

In [14]:
# %time: UMA execução, tempo de relógio (wall time) — bom para operações longas
%time sum(range(10**6))      # somar 0..999.999 — 10⁶ observações

CPU times: user 7.07 ms, sys: 13 μs, total: 7.08 ms
Wall time: 7.09 ms


499999500000

In [15]:
# %timeit: repete N vezes e reporta a melhor média — ideal para operações rápidas
%timeit sum(range(10**6))

7.31 ms ± 240 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [16]:
# O mesmo cálculo com NumPy: array contíguo + soma vetorizada em C
arr = np.arange(10**6)       # mesmo universo de números, agora em array
%timeit np.sum(arr)          # esperado: ~10x ou mais rápido que sum()
# Leitura: NumPy vence por ordem de grandeza — bloco contíguo de memória + código C
# compilado vs. percorrer objetos Python um a um (VanderPlas, Cap. 1).

91.1 μs ± 1.19 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 📝 Exercícios
Resolva no notebook. Tente antes de olhar a solução — cada `# EXERCÍCIO N` é
seguido pela célula `# SOLUÇÃO N` comentada (gabarito comentado completo na
apostila, Seção 7).

### Exercício 1 — Inflação acumulada de um período arbitrário
A fórmula do acumulado é genérica: produto dos $(1 + i_k/100)$ menos 1. Use a
lista `ipca_fev_jul` (fev–jul/2026, % a.m., BCB-SGS 433) e calcule o acumulado
**sem pandas**, com um loop `for` e `*=`. Checagem: o resultado deve ser
≈ **3,10%** — média geométrica de ~0,51% a.m., bem longe do pico de 2022-23.

In [17]:
# EXERCÍCIO 1 — seu código aqui

In [18]:
# SOLUÇÃO 1
ipca_fev_jul = [0.70, 0.88, 0.67, 0.58, 0.16, 0.07]  # % a.m., BCB-SGS 433
acumulado = 1.0
for i in ipca_fev_jul:          # loop simples — versão sem pandas
    acumulado *= (1 + i/100)
inflacao_6m = (acumulado - 1) * 100
print(f"Inflação acumulada fev-jul/2026: {inflacao_6m:.2f}%")

Inflação acumulada fev-jul/2026: 3.10%


### Exercício 2 — Boi gordo: do dólar para a arroba
Conversão do agrobusiness: 1 arroba = 15 kg de carcaça; 1 kg = 2,20462 lb →
1 arroba = **33,069 lb**. Com o futuro CME a US$ 2,20/lb e câmbio 5,1253,
calcule o preço em R$/arroba. Checagem de sanidade: ≈ **R$ 372,87/@** — se der
R$ 37 mil, o erro é de fator 100 (você multiplicou lb/kg duas vezes).

In [19]:
# EXERCÍCIO 2 — seu código aqui

In [20]:
# SOLUÇÃO 2
cambio = 5.1253                    # R$/US$ (04/09/2026, BCB-SGS 1)
boi_usd_lb = 2.20                  # US$/lb (futuro CME, exemplificativo)
arroba_brl = boi_usd_lb * 33.069 * cambio
print(f"Boi gordo: {brl(arroba_brl)} por arroba")   # ≈ R$ 372,87 por arroba

Boi gordo: R$ 372,87 por arroba


### Exercício 3 — Mini-tabela de preços do agro (dict)
Monte `precos = {"soja_sc60kg": 145.76, "boi_arroba": 372.87, "milho_sc60kg": 51.50}`
(preços em R$ calculados nesta aula). Depois: adicione a chave `"cambio": 5.1253`;
imprima o preço da soja com `brl()`; consulte `"cafe"` com `.get()` e padrão
`"não monitorado"`; liste as chaves finais.

In [21]:
# EXERCÍCIO 3 — seu código aqui

In [22]:
# SOLUÇÃO 3
precos = {"soja_sc60kg": 145.76, "boi_arroba": 372.87, "milho_sc60kg": 51.50}
precos["cambio"] = 5.1253          # atualiza/adiciona chave
print("preço soja:", brl(precos["soja_sc60kg"]))
print("preço café:", precos.get("cafe", "não monitorado"))
print("chaves:", list(precos.keys()))

preço soja: R$ 145,76
preço café: não monitorado
chaves: ['soja_sc60kg', 'boi_arroba', 'milho_sc60kg', 'cambio']


### Exercício 4 — Benchmark próprio (`%timeit`)
Compare três formas de somar 1 milhão de lançamentos contábeis
`valores = [1200.50 + i * 0.01 for i in range(1_000_000)]`: (a) `sum()` com
`%timeit`; (b) `np.sum()` sobre o array com `%timeit`; (c) loop manual com
`%%timeit` (célula inteira — o mágico deve ser a 1ª linha). O número exato
varia por máquina; a ordem de grandeza não: NumPy ≫ `sum()` ≫ loop.

In [23]:
# EXERCÍCIO 4 — seu código aqui
# Dica: %%timeit deve ser a PRIMEIRA linha da célula para cronometrar o bloco.

In [24]:
# SOLUÇÃO 4 — partes (a) e (b): cada %timeit cobre a instrução da sua linha
valores = [1200.50 + i * 0.01 for i in range(1_000_000)]  # lançamentos fictícios

%timeit sum(valores)                 # (a) Python puro

arr = np.array(valores)              # (b) NumPy vetorizado
%timeit np.sum(arr)                  # espera-se ~10x+ mais rápido que sum()

2.3 ms ± 3.64 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


175 μs ± 475 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [25]:
%%timeit
# SOLUÇÃO 4 — parte (c): loop manual cronometrado com %%timeit (célula inteira)
total_loop = 0.0
for x in valores:
    total_loop += x

9.63 ms ± 16 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## 📌 Resumo & para casa
- **Tipos governam operações**: `int` (sacas), `float` (preços — compare com `abs(a-b) < 1e-9`, nunca com `==`), `str` (texto), `bool` (decisões).
- **`/` sempre devolve float**; `//` e `%` para divisão inteira e resto; `**` para juros compostos; `+=`/`*=` acumulam sem reescrever a expressão.
- **Estruturas**: lista = série mutável por posição; tupla = registro imutável; dict = consulta por chave (o "PROCV mental") — escolha pela pergunta, não pelo hábito.
- **Acumulado de inflação = produto de (1+i/100) − 1**: ≈ 4,44% em 12 meses e ≈ 3,10% em fev–jul/2026 — o mesmo `PRODUTO()` do Excel, reutilizável para qualquer janela.
- **Medir antes de otimizar**: `%time` (uma rodada) vs `%timeit` (média de várias); `np.sum` vence `sum()` por ordem de grandeza.
- **Para casa**: refaça os 4 exercícios sem o gabarito, leia a apostila §4 (erros comuns) e traga dúvidas.

**Próximo passo (Aula 03)**: condicionais — `if/elif/else` decidindo "o projeto passa na TIR?" e alertas de câmbio.

**Referências da KB:**
- `kb/01_mckinney_python_for_data_analysis/15_appendix-python-language-essentials.md` — Apêndice: semântica, tipos escalares (Tabela A-2), tupla/lista/dict
- `kb/02_vanderplas_python_data_science_handbook/04_chapter-1-ipython-beyond-normal-python.md` — Cap. 1: "IPython Magic Commands", "Profiling and Timing Code"